In [2]:
# Unified inference pipeline (single-cell + pseudobulk)
import sys
import importlib
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path('/mnt/jack-5/amismailov/XGB')
if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

import preprocessor
importlib.reload(preprocessor)
from preprocessor import SingleCell

sc = SingleCell(
    path_length=str(PROJECT_DIR / 'df_gene_mapping.parquet'),
    path_features=str(PROJECT_DIR / 'features.json'),
    path_mrna=str(PROJECT_DIR / 'mRNA_names.json'),
    path_models=str(PROJECT_DIR / 'models'),
)

# Preload all staged models once (faster for repeated runs).
sc.load_models()


def return_to_external_inference(path_data, mapping_path=None):
    data = pd.read_csv(path_data, index_col=0)
    preds_ss = sc.predict_single_cell(
        data=data,
        mapping_path=mapping_path,
    )
    return preds_ss


def main_workflow(
    path_data,
    path_ss,
    path_clusters=None,
    path_bulk=None,
    mapping_path=None,
    barcode_col='barcode',
    cluster_col='cluster',
):
    ss_preds, pb_preds = sc.run_workflow(
        path_data=path_data,
        path_ss=path_ss,
        path_clusters=path_clusters,
        path_bulk=path_bulk,
        mapping_path=mapping_path,
        barcode_col=barcode_col,
        cluster_col=cluster_col,
    )
    return ss_preds, pb_preds

In [3]:
import os
path_to_datasets = '/mnt/jack-5/amismailov/miRNA_study/cancers'
datasets = os.listdir(path_to_datasets)

sample_dict = {
    'RCC' : 5,
    'breast' : 5,
    'col' : 5,
    'ovarian_met' : 6,
    'cervic' : 5,
    'DLBCL' : 6,
    'ICC' : 5,
    'pancreas' : 5,
    'LUAD_metastasis' : 7,
    'colorectal_met' : 5,
    'LUAD' : 5, 
    'breast_met' : 5,
    'HCC' : 5,
    'thyroid' : 6,
    'met_cholangiocarcinoma' : 5,
    'GC' : 5,
    'thyroid_met' : 5,
    'pbmc' : 5,
    'ovarian' : 6,
    'mel' : 5, 
    'ccRCC-BM' : 6, 
    'GCM' : 5,
    'cSCC' : 5
     
}

In [8]:
for dataset in datasets:
    print(f'Predicting {dataset}....')
    for id in range(1, sample_dict[dataset]+1):
        current_path = f'/mnt/jack-5/amismailov/miRNA_study/cancers/{dataset}/ML/sample_{id}.csv'
        path_predicted = f'/mnt/jack-5/amismailov/miRNA_study/cancers/{dataset}/ML/sample_{id}_XGB.csv'
        ss_preds = return_to_external_inference(path_data=current_path)
        ss_preds.to_csv(path_predicted)

Predicting RCC....


✔ Found length for 17392/17392 genes (100.00%)
Prediction of miRNAs expression (cascade by stages)..


stage_7: 100%|██████████| 100/100 [00:12<00:00,  8.16it/s]


KeyboardInterrupt: 

In [2]:
# Example: full workflow (single-cell + pseudobulk)
# Uncomment if you have clusters file with columns: barcode, cluster

# path_data = '/Users/neuropromotion/Downloads/inference/sample_1.csv'
# path_ss = '/Users/neuropromotion/Downloads/inference/sample_1_predicted_ss.csv'
# path_clusters = '/Users/neuropromotion/Downloads/inference/sample_1_clusters.csv'
# path_bulk = '/Users/neuropromotion/Downloads/inference/sample_1_predicted_pb.csv'

# ss_preds, pb_preds = main_workflow(
#     path_data=path_data,
#     path_ss=path_ss,
#     path_clusters=path_clusters,
#     path_bulk=path_bulk,
# )
# ss_preds.head()

In [1]:
import pandas as pd
df = pd.read_csv('/mnt/jack-5/amismailov/XGB/TEST_METRICS_XGB.csv', sep=';')
df.head()

,stage,miRNA,R2_K1,SPEARMAN_K1,R2_K2,SPEARMAN_K2,R2_K3,SPEARMAN_K3,R2_K4,SPEARMAN_K4,R2_K5,SPEARMAN_K5,R2_K10,SPEARMAN_K10
0,stage_1,hsa-let-7b,"0,81","0,86","0,75","0,88","0,77","0,89","0,73","0,90","0,74","0,87","0,65","0,93"
1,stage_1,hsa-let-7c,"0,45","0,62","0,65","0,75","0,72","0,82","0,75","0,85","0,72","0,86","0,79","0,86"
2,stage_1,hsa-let-7d,"0,78","0,86","0,75","0,89","0,76","0,89","0,77","0,89","0,79","0,92","0,78","0,94"
3,stage_1,hsa-let-7g,"0,60","0,79","0,69","0,88","0,76","0,91","0,62","0,89","0,79","0,94","0,78","0,95"
4,stage_1,hsa-let-7i,"0,77","0,74","0,89","0,78","0,93","0,80","0,90","0,83","0,92","0,80","0,84","0,84"


In [2]:
cols = ['miRNA', 'R2_K1', 'R2_K2', 'R2_K3', 'R2_K4', 'R2_K5', 'R2_K10']
df = df[cols]
df.head()

,miRNA,R2_K1,R2_K2,R2_K3,R2_K4,R2_K5,R2_K10
0,hsa-let-7b,"0,81","0,75","0,77","0,73","0,74","0,65"
1,hsa-let-7c,"0,45","0,65","0,72","0,75","0,72","0,79"
2,hsa-let-7d,"0,78","0,75","0,76","0,77","0,79","0,78"
3,hsa-let-7g,"0,60","0,69","0,76","0,62","0,79","0,78"
4,hsa-let-7i,"0,77","0,89","0,93","0,90","0,92","0,84"


In [3]:
df.index = df.pop('miRNA')
df = (
    df
    .replace(",", ".", regex=True)
    .apply(pd.to_numeric, errors="ignore")
)

/mnt/jack-1/tmp/ipykernel_3932868/1933833460.py:3: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df


In [4]:
df

,R2_K1,R2_K2,R2_K3,R2_K4,R2_K5,R2_K10
miRNA,,,,,,
hsa-let-7b,0.81,0.75,0.77,0.73,0.74,0.65
hsa-let-7c,0.45,0.65,0.72,0.75,0.72,0.79
hsa-let-7d,0.78,0.75,0.76,0.77,0.79,0.78
hsa-let-7g,0.60,0.69,0.76,0.62,0.79,0.78
hsa-let-7i,0.77,0.89,0.93,0.90,0.92,0.84
...,...,...,...,...,...,...
hsa-mir-95,-0.01,0.06,0.14,0.10,0.19,0.26
hsa-mir-1294,-0.00,-0.01,0.01,0.02,-0.05,0.08
hsa-mir-3127,-0.02,0.00,-0.02,0.04,-0.03,-0.02


In [8]:
df[(df['R2_K1'] < 0.4) & (df['R2_K2'] < 0.4) & (df['R2_K3'] < 0.4) & (df['R2_K4'] < 0.4) & (df['R2_K5'] < 0.4) & (df['R2_K10'] > 0.4)]

,R2_K1,R2_K2,R2_K3,R2_K4,R2_K5,R2_K10
miRNA,,,,,,
hsa-mir-210,0.10,0.18,0.20,0.17,0.31,0.81
hsa-mir-33a,0.00,0.08,0.12,0.30,0.37,0.73
hsa-mir-30e,0.10,0.24,0.33,0.38,0.20,0.65
hsa-mir-186,-0.03,0.10,0.11,0.20,0.20,0.43
hsa-mir-758,0.02,0.13,0.19,0.26,0.30,0.42
hsa-mir-1185-1,-0.05,0.08,-0.23,0.24,0.16,0.56
hsa-mir-584,0.02,0.10,0.20,0.18,0.10,0.51
hsa-mir-1910,0.02,0.11,0.27,0.27,0.11,0.54
hsa-mir-629,-0.01,-0.02,-0.01,0.09,0.06,0.43


In [26]:
(df.loc['hsa-let-7b'].values > 0.9).sum()

eligable_mirs = []
for mir in df.index:
    if (df.loc[mir].values > 0.4).sum() > 0:
        eligable_mirs.append(mir)
len(eligable_mirs)

147

In [31]:
#df.loc[eligable_mirs].to_csv('/mnt/jack-5/amismailov/XGB/XGB_ELIGABLE_MIRS.csv')

df.loc[eligable_mirs].to_excel(
    "/mnt/jack-5/amismailov/XGB/XGB_ELIGABLE_MIRS.xlsx", index=True
)

In [33]:
eligable_mirs

with open("/mnt/jack-5/amismailov/XGB/eligable_mirs.txt", "w") as f:
    for mir in eligable_mirs:
        # Важно: если в питоне они еще называются XGB-, можно заменить прямо тут:
        mir_fixed = mir.replace("XGB-", "hsa-")
        f.write(f"{mir_fixed}\n")

In [1]:
import pandas as pd
df = pd.read_parquet('/mnt/jack-5/amismailov/miRNA_study/ML_cancers/all_samples/RCC_S2.parquet')

In [4]:
import gc
gc.collect()


0